# Inference speed: RNA-FM vs RiNALMo

Compare wall-clock time for the operations the pipeline actually does:
1. **Single-sequence embedding** — latency per call
2. **Varying sequence length** — how does cost scale with length?
3. **The real win**: embed-once-and-slice (RiNALMo) vs re-embed-per-window (RNA-FM)
4. **Projected pipeline speedup** for short ncRNA + island alignment workloads

In [ ]:
import sys, time, random, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "modules" / "RNA-FM"))
sys.path.insert(0, str(REPO_ROOT / "modules" / "RiNALMo"))
sys.path.insert(0, str(REPO_ROOT / "modules" / "pipeline"))

import short_ncrna as sn
from pyrion import TwoBitAccessor
from pyrion.io.bed import read_bed12_file

SEED = 42
random.seed(SEED); np.random.seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", device)

In [ ]:
# --- Load both models ---
# NOTE: Both models have MPS compatibility patches.
# Optimizations are toggled via the rinalmo_optimizations.patch file.
import fm
fm_model, fm_alpha = fm.pretrained.rna_fm_t12()
fm_model.eval().to(device)
fm_bc = fm_alpha.get_batch_converter()

def emb_rnafm(seq: str) -> np.ndarray:
    rna = seq.upper().replace("T", "U")
    _, _, tk = fm_bc([("x", rna)])
    tk = tk.to(device)
    with torch.no_grad():
        out = fm_model(tk, repr_layers=[12])["representations"][12]
    return out[0, 1:1 + len(rna), :].cpu().float().numpy()

from rinalmo.pretrained import get_pretrained_model
rin_model, rin_alpha = get_pretrained_model(model_name="giga-v1")
rin_model = rin_model.to(device).eval()

def emb_rinalmo(seq: str) -> np.ndarray:
    rna = seq.upper().replace("T", "U")
    tokens = torch.tensor(
        rin_alpha.batch_tokenize([rna]), dtype=torch.int64, device=device
    )
    with torch.no_grad():
        out = rin_model(tokens)
    return out["representation"][0, 1:1 + len(rna), :].cpu().float().numpy()

MODELS = {
    "RNA-FM": {"fn": emb_rnafm, "dim": 640},
    "RiNALMo": {"fn": emb_rinalmo, "dim": 1280},
}
print(f"Both models loaded on {device}")

In [ ]:
# --- Generate test sequences ---
BASES = "AUGC"
rng = random.Random(SEED)

# Also load some real sequences for realistic benchmarks
accessor = TwoBitAccessor(str(REPO_ROOT / "input_data" / "2bit" / "hg38.2bit"))
bed_data = read_bed12_file(str(REPO_ROOT / "input_data" / "reference_annotation" / "hg38.input.w.tRNA.bed"))

real_seqs = []
for t in bed_data:
    s = sn._get_spliced_sequence(t, accessor)
    if s and "N" not in s.upper() and 40 <= len(s) <= 300:
        real_seqs.append(s.upper().replace("T", "U"))
    if len(real_seqs) >= 200:
        break
rng.shuffle(real_seqs)

print(f"Real sequences: {len(real_seqs)}, lengths {min(len(s) for s in real_seqs)}-{max(len(s) for s in real_seqs)} nt")

## Test 1: Single-sequence latency

Time a single embedding call, repeated N times, for fixed-length sequences.

In [ ]:
def bench_single(emb_fn, seqs, n_warmup=3):
    """Time single-sequence embedding calls. Returns list of per-call times."""
    for s in seqs[:n_warmup]:
        emb_fn(s)
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()

    times = []
    for s in seqs:
        t0 = time.perf_counter()
        emb_fn(s)
        if device.type == "cuda":
            torch.cuda.synchronize()
        elif device.type == "mps":
            torch.mps.synchronize()
        times.append(time.perf_counter() - t0)
    return times

# Fixed-length sequences for clean comparison
N_BENCH = 50
FIXED_LEN = 96  # typical pipeline window size
fixed_seqs = ["".join(rng.choices(BASES, k=FIXED_LEN)) for _ in range(N_BENCH)]

print(f"Benchmarking {N_BENCH} sequences of {FIXED_LEN} nt each...\n")
latency = {}
for mname, minfo in MODELS.items():
    ts = bench_single(minfo["fn"], fixed_seqs)
    latency[mname] = ts
    print(f"{mname:>10s}: {np.mean(ts)*1000:7.1f} ms/seq  "
          f"(std {np.std(ts)*1000:.1f}, min {np.min(ts)*1000:.1f}, max {np.max(ts)*1000:.1f})")

ratio = np.mean(latency["RiNALMo"]) / np.mean(latency["RNA-FM"])
print(f"\nRiNALMo / RNA-FM ratio: {ratio:.2f}x")

## Test 2: Scaling with sequence length

How does inference time grow with input length? Crucial for the embed-once strategy.

In [ ]:
LENGTHS = [48, 72, 96, 128, 192, 256, 384, 512, 768, 1024]
N_REP = 10

scaling_rows = []
for mname, minfo in MODELS.items():
    fn = minfo["fn"]
    # Warmup
    fn("".join(rng.choices(BASES, k=96)))
    if device.type in ("cuda", "mps"):
        getattr(torch, device.type).synchronize()

    for L in LENGTHS:
        seqs = ["".join(rng.choices(BASES, k=L)) for _ in range(N_REP)]
        times = []
        for s in seqs:
            t0 = time.perf_counter()
            fn(s)
            if device.type in ("cuda", "mps"):
                getattr(torch, device.type).synchronize()
            times.append(time.perf_counter() - t0)
        scaling_rows.append({
            "model": mname, "length": L,
            "mean_ms": np.mean(times) * 1000,
            "std_ms": np.std(times) * 1000,
        })
        print(f"  {mname:>10s} L={L:>5d}: {np.mean(times)*1000:7.1f} ms")

scale_df = pd.DataFrame(scaling_rows)
print("\nDone")

In [ ]:
# --- Scaling plot ---
fig, ax = plt.subplots(figsize=(7, 4))
for mname, color in [("RNA-FM", "C0"), ("RiNALMo", "C1")]:
    sub = scale_df[scale_df.model == mname]
    ax.errorbar(sub["length"], sub["mean_ms"], yerr=sub["std_ms"],
                marker="o", ms=5, label=mname, color=color, capsize=3)

ax.set_xlabel("Sequence length (nt)")
ax.set_ylabel("Inference time (ms)")
ax.set_title("Single-sequence embedding latency vs length")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout(); plt.show()

## Test 3: Embed-once-and-slice vs re-embed-per-window

This is the key pipeline question. Current RNA-FM approach:
- Extract N overlapping windows from a genomic region
- Embed each window separately (N forward passes)

RiNALMo alternative (context-stable embeddings):
- Embed the full region once (1 forward pass)
- Slice the embedding at window boundaries (numpy only, ~free)

In [ ]:
# Simulate the pipeline's windowed embedding vs embed-once
REGION_LENGTHS = [200, 400, 600, 800, 1000]
WINDOW_SIZE = 96
STRIDE = 4  # pipeline default for island scanning

strategy_rows = []

for region_len in REGION_LENGTHS:
    region_seq = "".join(rng.choices(BASES, k=region_len))
    n_windows = (region_len - WINDOW_SIZE) // STRIDE + 1

    # --- Strategy A: re-embed each window (both models) ---
    reembed_times = {}
    for mname, minfo in MODELS.items():
        fn = minfo["fn"]
        fn(region_seq[:WINDOW_SIZE])  # warmup
        if device.type in ("cuda", "mps"):
            getattr(torch, device.type).synchronize()

        t0 = time.perf_counter()
        for i in range(0, region_len - WINDOW_SIZE + 1, STRIDE):
            fn(region_seq[i:i + WINDOW_SIZE])
        if device.type in ("cuda", "mps"):
            getattr(torch, device.type).synchronize()
        reembed_time = time.perf_counter() - t0
        reembed_times[mname] = reembed_time

        strategy_rows.append({
            "region_len": region_len, "model": mname,
            "strategy": "re-embed", "n_windows": n_windows,
            "time_s": reembed_time,
        })

    # --- Strategy B: embed once + slice (RiNALMo only) ---
    fn = MODELS["RiNALMo"]["fn"]
    fn(region_seq)  # warmup
    if device.type in ("cuda", "mps"):
        getattr(torch, device.type).synchronize()

    t0 = time.perf_counter()
    full_emb = fn(region_seq)
    sliced = [full_emb[i:i + WINDOW_SIZE] for i in range(0, region_len - WINDOW_SIZE + 1, STRIDE)]
    if device.type in ("cuda", "mps"):
        getattr(torch, device.type).synchronize()
    slice_time = time.perf_counter() - t0

    strategy_rows.append({
        "region_len": region_len, "model": "RiNALMo",
        "strategy": "embed-once", "n_windows": n_windows,
        "time_s": slice_time,
    })

    print(f"Region {region_len:>5d} nt ({n_windows:>3d} windows):  "
          f"RNA-FM re-embed={reembed_times['RNA-FM']:.2f}s  "
          f"RiNALMo re-embed={reembed_times['RiNALMo']:.2f}s  "
          f"RiNALMo embed-once={slice_time:.3f}s")

strat_df = pd.DataFrame(strategy_rows)

In [ ]:
# --- Strategy comparison plot ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: absolute time
ax = axes[0]
for label, ls, color in [
    ("RNA-FM re-embed", "-", "C0"),
    ("RiNALMo re-embed", "--", "C1"),
    ("RiNALMo embed-once", "-", "C2"),
]:
    parts = label.split(" ", 1)
    sub = strat_df[(strat_df.model == parts[0]) & (strat_df.strategy == parts[1])]
    ax.plot(sub["region_len"], sub["time_s"], marker="o", ms=5, ls=ls, color=color, label=label)

ax.set_xlabel("Region length (nt)")
ax.set_ylabel("Wall time (s)")
ax.set_title(f"Window={WINDOW_SIZE}nt, stride={STRIDE}nt")
ax.legend(frameon=False, fontsize=9)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# Right: speedup factor
ax = axes[1]
for baseline_model, baseline_label, color in [
    ("RNA-FM", "vs RNA-FM re-embed", "C0"),
    ("RiNALMo", "vs RiNALMo re-embed", "C1"),
]:
    base = strat_df[(strat_df.model == baseline_model) & (strat_df.strategy == "re-embed")]
    fast = strat_df[(strat_df.model == "RiNALMo") & (strat_df.strategy == "embed-once")]
    if len(base) == len(fast):
        speedup = base["time_s"].values / fast["time_s"].values
        ax.plot(base["region_len"].values, speedup, marker="o", ms=5, color=color, label=baseline_label)

ax.set_xlabel("Region length (nt)")
ax.set_ylabel("Speedup (x)")
ax.set_title("RiNALMo embed-once speedup")
ax.axhline(1, color="grey", lw=0.5, ls=":")
ax.legend(frameon=False, fontsize=9)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

plt.tight_layout(); plt.show()

## Test 4: Real-world pipeline throughput estimate

Project the speedup to a typical CURIA run:
- ~2800 short ncRNAs (embed ref + query, each ~50-160 nt)
- ~200 island pairs (windowed embedding, regions ~100-800 nt)

In [ ]:
# --- Throughput on real sequences ---
N_THROUGHPUT = min(100, len(real_seqs))
test_seqs = real_seqs[:N_THROUGHPUT]

print(f"Throughput benchmark: {N_THROUGHPUT} real ncRNA sequences")
print(f"Length range: {min(len(s) for s in test_seqs)}-{max(len(s) for s in test_seqs)} nt\n")

for mname, minfo in MODELS.items():
    fn = minfo["fn"]
    # Warmup
    fn(test_seqs[0])
    if device.type in ("cuda", "mps"):
        getattr(torch, device.type).synchronize()

    t0 = time.perf_counter()
    for s in test_seqs:
        fn(s)
    if device.type in ("cuda", "mps"):
        getattr(torch, device.type).synchronize()
    elapsed = time.perf_counter() - t0

    print(f"{mname:>10s}: {elapsed:.2f}s total, {elapsed/N_THROUGHPUT*1000:.1f} ms/seq, "
          f"{N_THROUGHPUT/elapsed:.1f} seq/s")

In [ ]:
# --- Projected pipeline time ---
print("=" * 70)
print("PROJECTED PIPELINE SPEEDUP")
print("=" * 70)

# Get per-sequence latencies from the fixed-length benchmark
rnafm_ms = np.mean(latency["RNA-FM"]) * 1000
rinalmo_ms = np.mean(latency["RiNALMo"]) * 1000

# Typical pipeline numbers (from preprint_results/hg38_vs_mm39)
N_SHORT = 2800   # short ncRNA predictions
N_ISLANDS = 200  # island pairs
AVG_ISLAND_LEN = 300  # nt
N_WINDOWS_PER_ISLAND = (AVG_ISLAND_LEN - WINDOW_SIZE) // STRIDE + 1

print(f"\nAssumptions:")
print(f"  Short ncRNAs: {N_SHORT} (ref + query = {N_SHORT * 2} embeddings)")
print(f"  Island pairs: {N_ISLANDS} (avg {AVG_ISLAND_LEN} nt, {N_WINDOWS_PER_ISLAND} windows each)")
print(f"  Per-embedding latency: RNA-FM={rnafm_ms:.1f}ms, RiNALMo={rinalmo_ms:.1f}ms")

# Strategy 1: RNA-FM re-embed everything (current)
short_rnafm = N_SHORT * 2 * rnafm_ms / 1000  # seconds
island_rnafm = N_ISLANDS * 2 * N_WINDOWS_PER_ISLAND * rnafm_ms / 1000
total_rnafm = short_rnafm + island_rnafm

# Strategy 2: RiNALMo re-embed everything (naive swap)
short_rinalmo = N_SHORT * 2 * rinalmo_ms / 1000
island_rinalmo_reemb = N_ISLANDS * 2 * N_WINDOWS_PER_ISLAND * rinalmo_ms / 1000
total_rinalmo_naive = short_rinalmo + island_rinalmo_reemb

# Strategy 3: RiNALMo embed-once (the win)
# Short ncRNAs: still 1 embed per sequence
# Islands: 1 embed per island (not per window)
island_rinalmo_once = N_ISLANDS * 2 * rinalmo_ms / 1000  # just 2 embeds per pair
total_rinalmo_smart = short_rinalmo + island_rinalmo_once

print(f"\n{'Strategy':>30s}  {'Short':>8s}  {'Islands':>8s}  {'Total':>8s}  {'Speedup':>8s}")
print("-" * 70)
print(f"{'RNA-FM re-embed (current)':>30s}  {short_rnafm:>7.1f}s  {island_rnafm:>7.1f}s  {total_rnafm:>7.1f}s  {'1.0x':>8s}")
print(f"{'RiNALMo re-embed (naive)':>30s}  {short_rinalmo:>7.1f}s  {island_rinalmo_reemb:>7.1f}s  {total_rinalmo_naive:>7.1f}s  {total_rnafm/total_rinalmo_naive:>7.1f}x")
print(f"{'RiNALMo embed-once (smart)':>30s}  {short_rinalmo:>7.1f}s  {island_rinalmo_once:>7.1f}s  {total_rinalmo_smart:>7.1f}s  {total_rnafm/total_rinalmo_smart:>7.1f}x")

print(f"\nIsland alignment speedup alone (embed-once vs re-embed):")
print(f"  vs RNA-FM:   {island_rnafm / island_rinalmo_once:.0f}x")
print(f"  vs RiNALMo:  {island_rinalmo_reemb / island_rinalmo_once:.0f}x")

## Summary

In [ ]:
print("=" * 70)
print("SPEED COMPARISON SUMMARY")
print("=" * 70)

print(f"\n1. Single-sequence latency ({FIXED_LEN} nt):")
print(f"   RNA-FM:   {np.mean(latency['RNA-FM'])*1000:.1f} ms")
print(f"   RiNALMo:  {np.mean(latency['RiNALMo'])*1000:.1f} ms")
print(f"   Ratio:    {ratio:.2f}x")

print(f"\n2. Scaling:")
for L in [96, 256, 512, 1024]:
    fm_t = scale_df[(scale_df.model == "RNA-FM") & (scale_df.length == L)]["mean_ms"]
    rin_t = scale_df[(scale_df.model == "RiNALMo") & (scale_df.length == L)]["mean_ms"]
    if len(fm_t) > 0 and len(rin_t) > 0:
        print(f"   L={L:>5d}: RNA-FM={fm_t.values[0]:.1f}ms, RiNALMo={rin_t.values[0]:.1f}ms, ratio={rin_t.values[0]/fm_t.values[0]:.2f}x")

print(f"\n3. Embed-once speedup (region={REGION_LENGTHS[-1]}nt, window={WINDOW_SIZE}, stride={STRIDE}):")
base_fm = strat_df[(strat_df.model == "RNA-FM") & (strat_df.strategy == "re-embed") & (strat_df.region_len == REGION_LENGTHS[-1])]["time_s"]
fast = strat_df[(strat_df.model == "RiNALMo") & (strat_df.strategy == "embed-once") & (strat_df.region_len == REGION_LENGTHS[-1])]["time_s"]
if len(base_fm) > 0 and len(fast) > 0:
    print(f"   RNA-FM re-embed:      {base_fm.values[0]:.2f}s")
    print(f"   RiNALMo embed-once:   {fast.values[0]:.3f}s")
    print(f"   Speedup:              {base_fm.values[0]/fast.values[0]:.0f}x")

print(f"\n4. Projected full pipeline (hg38->mm39 scale):")
print(f"   Current (RNA-FM):              {total_rnafm:.0f}s")
print(f"   RiNALMo embed-once:            {total_rinalmo_smart:.0f}s")
print(f"   Overall speedup:               {total_rnafm/total_rinalmo_smart:.1f}x")